# HỆ THỐNG TRỢ LÝ AI ĐA DỤNG TRÍCH XUẤT NGUYÊN VĂN (UNIVERSAL EXTRACTIVE RAG SYSTEM)

### Mô tả dự án:
Dự án xây dựng một hệ thống **General-Purpose Extractive RAG (Hệ thống RAG trích xuất đa năng)** chuyên sâu cho các tài liệu phức tạp (như Văn bản Pháp luật, Sổ tay Kỹ thuật, Báo cáo Tài chính). Hệ thống giải quyết triệt để bài toán **ảo giác (hallucination)** và **trích dẫn sai lệch nguồn ngữ cảnh** bằng cách ép buộc mô hình ngôn ngữ lớn (LLM) trích xuất nguyên văn (verbatim) các điều khoản trực tiếp từ tài liệu gốc, kết hợp với khả năng tự động định vị tiêu đề (Điều/Chương/Mục) và số trang vật lý chính xác.

### Kiến trúc công nghệ nâng cao (SOTA Architecture):
1. **Hierarchical Parent-Child Chunking:** Tách biệt lớp tìm kiếm mịn (Child) và lớp đọc hiểu sâu rộng (Parent).
2. **Hybrid Search (Ensemble Retrieval):** Kết hợp tìm kiếm từ khóa truyền thống (**BM25**) và tìm kiếm ngữ nghĩa (**Dense Vector - SBERT**).
3. **Cross-Encoder Reranking:** Tái xếp hạng các ứng viên bằng mô hình **`BAAI/bge-reranker-base`** nhằm tối ưu hóa độ phủ của văn bản đầu vào.
4. **4-bit Quantized Local LLM:** Sử dụng **`Qwen2.5-7B-Instruct`** chạy offline trên đơn GPU T4 để tối ưu hóa tài nguyên VRAM và tốc độ phản hồi.
5. **Automated Multi-Metric Evaluation:** Đánh giá hệ thống trên diện rộng bằng 4 chỉ số học thuật tiêu chuẩn: **Semantic Cosine Similarity, ROUGE-L (DP), BLEU-2 (NLTK), và Token F1-Score (SQuAD)**.

## 1. THIẾT LẬP MÔI TRƯỜNG & THƯ VIỆN HỆ THỐNG
Tiến hành cài đặt các thư viện lõi phục vụ cho hạ tầng RAG, lượng tử hóa mô hình và tính toán chỉ số đánh giá.
- `langchain-huggingface` & `langchain-chroma`: Quản lý pipeline RAG chuẩn mới.
- `bitsandbytes` & `accelerate`: Hỗ trợ nén lượng tử 4-bit và tối ưu hóa bộ nhớ GPU.
- `rank_bm25`: Công cụ tìm kiếm từ khóa truyền thống.

In [1]:
!pip install -q langchain langchain-community langchain-text-splitters pypdf chromadb sentence-transformers transformers accelerate bitsandbytes gradio langchain-huggingface langchain-chroma rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 103.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 84.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━

## 2. KIẾN TRÚC CHA-CON (PARENT-CHILD) & BỘ TÌM KIẾM LAI (HYBRID RETRIEVER)

### Ý tưởng giải quyết bài toán Chunking:
Trong văn bản pháp luật hoặc tài liệu kỹ thuật dài, việc cắt chữ cố định (Static Chunking) sẽ làm vỡ cấu trúc của các Điều khoản. Chúng ta giải quyết bài toán này bằng kiến trúc **Parent-Child**:
- **Lớp Con (Child Chunks - 350 ký tự):** Dùng để Vector hóa. Kích thước nhỏ giúp bộ tìm kiếm định vị chính xác từ khóa và ngữ nghĩa của câu hỏi.
- **Lớp Cha (Parent Chunks - 1600 ký tự):** Chứa trọn vẹn ngữ cảnh của Điều luật xung quanh đoạn Con. Lớp Cha sẽ được nhúng sẵn thuộc tính vật lý chính xác là `[Trang X]` của tài liệu để làm mỏ neo trích dẫn.

### Tìm kiếm Lai (Hybrid Search):
Sử dụng song song **BM25Retriever** (để không bỏ sót các số hiệu điều luật, tên riêng, thuật ngữ kỹ thuật) và **Chroma Vector Retriever** (để hiểu ngữ nghĩa câu hỏi), sau đó tự động gộp kết quả và loại bỏ trùng lặp bằng thuật toán tối ưu hóa trong Python.

In [2]:
import os
import shutil
import uuid
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

PDF_PATH = "/kaggle/input/datasets/phtranth/luatgiaothong-noneocr-2/luatgiaothong.pdf" # ĐƯỜNG DẪN FILE PDF MUỐN LÀM
DB_PATH = "/kaggle/working/chromadb_hybrid_parent_child_v4"

print("1. Đang phân tách tài liệu thành cấu trúc Parent-Child...")
if os.path.exists(DB_PATH):
    shutil.rmtree(DB_PATH)

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1600, chunk_overlap=300)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=50)

parent_docs = parent_splitter.split_documents(pages)
child_docs = []
parent_store = {} 

for p_doc in parent_docs:
    parent_id = str(uuid.uuid4())
    
    page_num = p_doc.metadata.get("page", 0) + 1
    
    parent_text_with_page = f"[Trang {page_num}]\n{p_doc.page_content}"
    parent_store[parent_id] = parent_text_with_page
    
    sub_chunks = child_splitter.split_text(p_doc.page_content)
    for sub_text in sub_chunks:
        child_doc = Document(
            page_content=sub_text,
            metadata={"parent_id": parent_id}
        )
        child_docs.append(child_doc)

print("2. Đang cấu hình bộ tìm kiếm từ khóa BM25 và bộ tìm kiếm Vector...")
# BỘ TÌM KIẾM 1: Tìm kiếm Ngữ nghĩa (Vector Search)
embeddings = HuggingFaceEmbeddings(model_name="keepitreal/vietnamese-sbert", model_kwargs={'device': 'cuda'})
vectordb = Chroma.from_documents(documents=child_docs, embedding=embeddings, persist_directory=DB_PATH)
vector_retriever = vectordb.as_retriever(search_kwargs={"k": 8}) 

# BỘ TÌM KIẾM 2: Tìm kiếm từ khóa chính xác (BM25)
bm25_retriever = BM25Retriever.from_documents(child_docs)
bm25_retriever.k = 8

print(f"-> Khởi tạo thành công các bộ tìm kiếm độc lập!")
print(f"Tổng số Chunks Con: {len(child_docs)} | Tổng số Chunks Cha: {len(parent_docs)}")

/tmp/ipykernel_58/3341709320.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


1. Đang phân tách tài liệu thành cấu trúc Parent-Child...
2. Đang cấu hình bộ tìm kiếm từ khóa BM25 và bộ tìm kiếm Vector...


/tmp/ipykernel_58/3341709320.py:46: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="keepitreal/vietnamese-sbert", model_kwargs={'device': 'cuda'})


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

-> Khởi tạo thành công các bộ tìm kiếm độc lập!
Tổng số Chunks Con: 640 | Tổng số Chunks Cha: 138


## 3. KHỞI TẠO NÃO BỘ LLM 7B TRÊN ĐƠN GPU (INFERENCE OPTIMIZATION)
- Sử dụng mô hình **Qwen2.5-7B-Instruct** lượng tử hóa 4-bit (`load_in_4bit=True`). Bản nén này chỉ chiếm khoảng 5.5GB VRAM, giúp chạy hoàn hảo trên duy nhất GPU 0 của Kaggle (tránh được độ trễ truyền tải dữ liệu PCIe khi chia nhỏ sang Multi-GPU).
- Cấu hình tham số sinh chữ nghiêm ngặt: `do_sample=False` để tắt hoàn toàn sự sáng tạo ngẫu nhiên, kết hợp cấu hình bộ ngắt dừng `eos_token_id` để ngăn chặn lỗi lặp từ vô hạn (looping).

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline

model_id = "Qwen/Qwen2.5-7B-Instruct"

print("Đang cấu hình Lượng tử hóa 4-bit...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Đang tải mô hình {model_id} từ HuggingFace (Khoảng 5.5GB)...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=quant_config, 
    device_map={"": 0} 
)

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|im_end|>")
]

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=450, 
    do_sample=False,        
    repetition_penalty=1.0, 
    eos_token_id=terminators,
    return_full_text=False
)
llm = HuggingFacePipeline(pipeline=pipe)
print("-> Khởi tạo LLM 7B thành công!")

Đang cấu hình Lượng tử hóa 4-bit...
Đang tải mô hình Qwen/Qwen2.5-7B-Instruct từ HuggingFace (Khoảng 5.5GB)...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'eos_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


-> Khởi tạo LLM 7B thành công!


## 4. BỘ TÁI XẾP HẠNG (RERANKER) & LUỒNG RAG TRÍCH XUẤT NGUYÊN VĂN TỰ ĐỘNG

### Reranking bằng Cross-Encoder:
Bộ tìm kiếm Lai lôi về tối đa 16 đoạn văn ứng viên (để không bị sót thông tin). Sau đó, mô hình Cross-Encoder **`BAAI/bge-reranker-base`** siêu nhẹ sẽ tính toán lại mức độ liên quan thực tế giữa Câu hỏi và các đoạn văn, lọc ra đúng 3 đoạn chuẩn xác nhất đưa cho LLM đọc. Điều này giúp loại bỏ hoàn toàn các lỗi báo "Không tìm thấy" do Vector Search bị nhiễu.

### Prompt Trích xuất Tiêu đề tự động (Dynamic Citation Prompt):
Sử dụng trí thông minh của mô hình 7B để tự động quét nội dung đoạn đọc:
- Nếu đoạn văn có ghi rõ số hiệu chương mục (ví dụ: "Điều 50"), AI sẽ tự động trích dẫn: `Theo quy định tại Điều 50 (Trang X)...`
- Nếu đoạn văn không ghi rõ đề mục, AI tự động lùi về dùng số trang vật lý cứng: `Theo quy định tại Trang X...`
- Ép buộc mô hình **sao chép y nguyên văn (verbatim copy-paste)** quy định gốc từ PDF, cấm tự ý tóm tắt hay sửa đổi câu chữ.

In [4]:
import torch
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

reranker_id = "BAAI/bge-reranker-base"
print(f"Đang tải mô hình Reranker từ HuggingFace: {reranker_id}...")
reranker_tokenizer = AutoTokenizer.from_pretrained(reranker_id)
reranker_model = AutoModelForSequenceClassification.from_pretrained(reranker_id).to("cuda:0")
print("-> Tải Reranker thành công!")

def retrieve_hybrid_reranked(question, k_top=3):
    bm25_docs = bm25_retriever.invoke(question)
    vector_docs = vector_retriever.invoke(question)
    
    candidates = []
    seen_content = set()
    for doc in bm25_docs + vector_docs:
        if doc.page_content not in seen_content:
            seen_content.add(doc.page_content)
            candidates.append(doc)
            
    if not candidates:
        return []
        
    pairs = [[question, doc.page_content] for doc in candidates]
    with torch.no_grad():
        inputs = reranker_tokenizer(pairs, padding=True, truncation=True, return_tensors='pt').to("cuda:0")
        scores = reranker_model(**inputs).logits.view(-1).float().cpu().numpy()
        
    ranked_indices = np.argsort(scores)[::-1]
    ranked_docs = [candidates[idx] for idx in ranked_indices]
    
    return ranked_docs[:k_top]

def query_rag_general(question):
    # Bước A: Tìm kiếm Lai và Rerank lọc ra 3 ứng viên tốt nhất
    child_docs = retrieve_hybrid_reranked(question, k_top=3)
    
    seen_parent_ids = set()
    parent_contexts = []
    
    # Bước B: Lấy tài liệu Cha tương ứng 
    for doc in child_docs:
        p_id = doc.metadata.get("parent_id")
        if p_id and p_id not in seen_parent_ids:
            seen_parent_ids.add(p_id)
            parent_contexts.append(parent_store[p_id]) 
            
    context = "\n\n".join(parent_contexts)
    
    # Bước C: Prompt - AI tự đọc và trích xuất tiêu đề
    messages = [
        {
            "role": "system", 
            "content": """Bạn là một trợ lý ảo chuyên nghiệp có nhiệm vụ trích xuất tài liệu pháp lý và kỹ thuật.

QUY TẮC TRÍCH DẪN BẮT BUỘC:
1. Đọc kỹ văn bản được cung cấp để tìm quy định trực tiếp giải quyết câu hỏi.
2. Bạn phải bắt đầu câu trả lời bằng một trích dẫn nguồn cực kỳ chính xác. Hãy đọc kỹ văn bản để tìm số hiệu tiêu đề (Ví dụ: "Điều X", "Chương Y", "Mục Z") và đối chiếu với nhãn số trang "[Trang Y]" được ghi ở đầu văn bản đó.
   - Trường hợp 1: Nếu trong văn bản có ghi rõ số hiệu tiêu đề (ví dụ "Điều 50"), hãy dẫn nguồn theo dạng: "Theo quy định tại Điều 50 (Trang Y) của tài liệu:"
   - Trường hợp 2: Nếu trong văn bản không ghi rõ số hiệu tiêu đề cụ thể nào, hãy dùng số trang để dẫn nguồn: "Theo quy định tại Trang Y của tài liệu:"
3. Tiếp sau câu trích dẫn nguồn, hãy SAO CHÉP Y NGUYÊN VĂN quy định đó từ tài liệu gốc. Định dạng đầu ra bắt buộc:
   "Theo quy định tại <Điều/Chương/Mục X> (Trang Y) của tài liệu:
   - [Đoạn văn sao chép nguyên văn]"
4. CẤM viết lời chào hỏi, không viết lời dẫn dắt khác, không ký tên ở cuối. Chỉ in ra đúng cấu trúc yêu cầu.
5. Nếu tài liệu hoàn toàn không chứa câu trả lời trực tiếp cho câu hỏi, chỉ trả lời duy nhất: "Không tìm thấy thông tin phù hợp trong tài liệu" và không giải thích gì thêm."""
        },
        {
            "role": "user", 
            "content": f"Tài liệu:\n{context}\n\nCâu hỏi: {question}"
        }
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = pipe(prompt)
    response = output[0]['generated_text'].strip()
    return response.split("<|im_start|>assistant")[-1].strip()

print("Hàm RAG Lai + Rerank + Trích dẫn nguồn thông minh đã cấu hình thành công!")

Đang tải mô hình Reranker từ HuggingFace: BAAI/bge-reranker-base...


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-> Tải Reranker thành công!
Hàm RAG Lai + Rerank + Trích dẫn nguồn thông minh đã cấu hình thành công!


## 5. TỰ ĐỘNG TẠO ĐỀ THI KIỂM THỬ SẠCH (AUTOMATED SYNTHETIC QA GENERATION)
- Để đánh giá hệ thống một cách khách quan trên diện rộng (thống kê học thuật), một thuật toán tự động hóa để LLM tự đọc hiểu tài liệu và soạn thảo ra bộ đề thi (gồm câu hỏi và đáp án Ground Truth).
- **Quy tắc câu hỏi tự nhiên:** Sử dụng kỹ thuật Prompt nâng cao để cấm AI đưa các từ khóa chỉ dẫn nguồn (như "Trang X", "theo tài liệu") vào câu hỏi, bảo đảm câu hỏi sinh ra tự nhiên 100% như người dùng gõ thực tế.
- **Quy tắc đáp án mẫu trích xuất:** Ép buộc câu trả lời mẫu phải sử dụng đúng cấu trúc trích dẫn nguyên văn để làm thước đo đối chiếu sòng phẳng cho chatbot ở bước sau.
- Sử dụng vòng lặp `while` để kiểm duyệt và đảm bảo thu thập đủ chính xác số lượng câu hỏi chất lượng cao yêu cầu.

In [5]:
import json
import random

NUM_QUESTIONS = 100 

print(f"=== TIẾN TRÌNH TẠO TỰ ĐỘNG ĐÚNG {NUM_QUESTIONS} CÂU HỎI TRÍCH DẪN THỰC TẾ ===")

all_parent_texts = list(parent_store.values())
random.shuffle(all_parent_texts)

generated_dataset = []
attempt = 0

while len(generated_dataset) < NUM_QUESTIONS and attempt < len(all_parent_texts):
    chunk = all_parent_texts[attempt]
    attempt += 1
    
    print(f"Đang phân tích đoạn văn {attempt}/{len(all_parent_texts)} | Đã tạo thành công: {len(generated_dataset)}/{NUM_QUESTIONS} câu...")
    
    messages = [
        {
            "role": "system", 
            "content": """Bạn là một chuyên gia soạn đề thi trắc nghiệm pháp luật và kỹ thuật nâng cao.
Nhiệm vụ của bạn là đọc đoạn văn được cung cấp. Lưu ý: Ở ngay đầu đoạn văn luôn có nhãn chỉ rõ số trang vật lý dạng "[Trang Y]".

Hãy tạo ra:
1. Một câu hỏi cụ thể (CH:) hỏi về một quy định cụ thể được nêu trong đoạn văn.
   QUY TẮC CẤM ĐỐI VỚI CÂU HỎI (CH:): 
   - CẤM TUYỆT ĐỐI đưa các từ như "Trang X", "theo tài liệu", "theo quy định tại", "theo trang" vào trong câu hỏi.
   - Câu hỏi phải hoàn toàn tự nhiên, ngắn gọn như một người dân bình thường đang thắc mắc gõ vào ô chat. 
   - Câu hỏi bắt buộc kết thúc bằng dấu hỏi chấm.

2. Câu trả lời chuẩn (TL:): Bạn phải định dạng chính xác theo cấu trúc sau (lấy số trang X từ nhãn ở đầu đoạn văn):
   "Theo thông tin trích xuất từ Trang X của tài liệu:
   - [Đoạn văn sao chép nguyên văn quy định đó từ tài liệu gốc]"

TUYỆT ĐỐI không viết lại, không tóm tắt, không được thay đổi bất kỳ từ ngữ hay ký tự nào của tài liệu gốc bên dưới nhãn trang.

Định dạng đầu ra bắt buộc viết chính xác như sau:
CH: [Câu hỏi tự nhiên của bạn]
TL: [Câu trả lời chuẩn theo đúng cấu trúc yêu cầu]"""
        },
        {
            "role": "user", 
            "content": f"Đoạn văn:\n{chunk}"
        }
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    pipe.model.config.max_new_tokens = 350 
    output = pipe(prompt)
    raw_text = output[0]['generated_text'].strip()
    
    try:
        lines = raw_text.split("\n")
        q_text, a_text = "", ""
        
        for line in lines:
            line_clean = line.replace("*", "").strip().lower()
            if "ch:" in line_clean or "câu hỏi:" in line_clean or "question:" in line_clean:
                parts = line.split(":", 1)
                if len(parts) > 1:
                    q_text = parts[1].replace("*", "").strip()
            elif "tl:" in line_clean or "câu trả lời:" in line_clean or "answer:" in line_clean:
                parts = line.split(":", 1)
                if len(parts) > 1:
                    idx_colon = line.find(":")
                    a_text = line[idx_colon+1:].replace("*", "").strip()
                    
                    idx_line = lines.index(line)
                    for remaining_line in lines[idx_line+1:]:
                        a_text += "\n" + remaining_line.replace("*", "").strip()
                    break
        
        if q_text and a_text and q_text.endswith("?"):
            generated_dataset.append({
                "question": q_text,
                "ground_truth": a_text
            })
    except Exception as e:
        continue

dataset_file = "/kaggle/working/golden_dataset.json"
with open(dataset_file, "w", encoding="utf-8") as f:
    json.dump(generated_dataset, f, ensure_ascii=False, indent=4)

print(f"\n✅ HOÀN TẤT! Đã tạo thành công chính xác {len(generated_dataset)} câu hỏi trích xuất thực tế chất lượng cao!")

Both `max_new_tokens` (=450) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== TIẾN TRÌNH TẠO TỰ ĐỘNG ĐÚNG 100 CÂU HỎI TRÍCH DẪN THỰC TẾ ===
Đang phân tích đoạn văn 1/138 | Đã tạo thành công: 0/100 câu...


Both `max_new_tokens` (=450) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Đang phân tích đoạn văn 2/138 | Đã tạo thành công: 1/100 câu...


Both `max_new_tokens` (=450) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Đang phân tích đoạn văn 98/138 | Đã tạo thành công: 97/100 câu...


Both `max_new_tokens` (=450) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Đang phân tích đoạn văn 99/138 | Đã tạo thành công: 98/100 câu...


Both `max_new_tokens` (=450) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Đang phân tích đoạn văn 100/138 | Đã tạo thành công: 99/100 câu...

✅ HOÀN TẤT! Đã tạo thành công chính xác 100 câu hỏi trích xuất thực tế chất lượng cao!


## 6. PIPELINE CHẤM ĐIỂM TỰ ĐỘNG & BÁO CÁO HỌC THUẬT (DASHBOARD)
Tiến hành chạy kiểm thử toàn diện trên bộ đề thi vừa tạo. Hệ thống tự động đo lường hiệu năng của chatbot bằng 4 chỉ số khoa học cao cấp và dựng lên bảng Dashboard báo cáo trực quan bằng HTML/CSS:
1. **Semantic Similarity (SBERT):** Đo lường mức độ trùng khớp ý nghĩa ngữ nghĩa (Semantics) của câu trả lời thông qua khoảng cách góc Cosine của Vector.
2. **ROUGE-L (LCS-F1):** Đo lường chuỗi từ khớp liên tục dài nhất giữa hai câu (Đánh giá khả năng trích xuất chính xác cấu trúc từ vựng).
3. **BLEU Score (BLEU-2):** Đánh giá độ chính xác của các cụm từ (N-gram) xuất hiện trong câu trả lời mẫu.
4. **Token F1-Score (SQuAD Standard):** Chỉ số chuẩn đo lường độ phủ của từ vựng (Precision & Recall ở cấp độ Token).

In [6]:
import json
import time
import numpy as np
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from IPython.display import display, HTML

nltk.download('punkt', quiet=True)

# GIẢI THUẬT TÍNH ĐIỂM CHUẨN KHOA HỌC
def lcs(x, y):
    m, n = len(x), len(y)
    L = [[0] * (n + 1) for i in range(m + 1)]
    for i in range(m + 1):
        for j in range(n + 1):
            if i == 0 or j == 0: L[i][j] = 0
            elif x[i-1] == y[j-1]: L[i][j] = L[i-1][j-1] + 1
            else: L[i][j] = max(L[i-1][j], L[i][j-1])
    return L[m][n]

def calculate_rouge_l(truth, prediction):
    truth_tokens = truth.lower().split()
    pred_tokens = prediction.lower().split()
    if not truth_tokens or not pred_tokens: return 0.0
    lcs_len = lcs(truth_tokens, pred_tokens)
    precision = lcs_len / len(pred_tokens)
    recall = lcs_len / len(truth_tokens)
    if (precision + recall) == 0: return 0.0
    return ((2 * precision * recall) / (precision + recall)) * 100

def calculate_bleu(truth, prediction):
    truth_tokens = [truth.lower().split()]
    pred_tokens = prediction.lower().split()
    smooth = SmoothingFunction().method1
    return sentence_bleu(truth_tokens, pred_tokens, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth) * 100

def calculate_token_f1(truth, prediction):
    truth_tokens = truth.lower().split()
    pred_tokens = prediction.lower().split()
    common = set(truth_tokens) & set(pred_tokens)
    num_same = len(common)
    if num_same == 0: return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)
    return ((2 * precision * recall) / (precision + recall)) * 100

def calculate_semantic_score(text1, text2):
    emb1 = np.array(embeddings.embed_query(text1))
    emb2 = np.array(embeddings.embed_query(text2))
    dot_product = np.dot(emb1, emb2)
    norm1, norm2 = np.linalg.norm(emb1), np.linalg.norm(emb2)
    return float((dot_product / (norm1 * norm2)) * 100)

# CHẠY CHẤM ĐIỂM TỰ ĐỘNG
dataset_file = "/kaggle/working/golden_dataset.json"
with open(dataset_file, "r", encoding="utf-8") as f:
    test_dataset = json.load(f)

print(f"=== ĐANG CHẠY CHẤM ĐIỂM TRÊN {len(test_dataset)} CÂU HỎI ===")
report = []

for idx, item in enumerate(test_dataset):
    q = item["question"]
    gt = item["ground_truth"]
    
    start_time = time.time()
    # Gọi bộ tìm kiếm Lai kết hợp Rerank đã định nghĩa ở CELL 4
    child_docs = retrieve_hybrid_reranked(q, k_top=3) 
    bot_answer = query_rag_general(q)
    latency = time.time() - start_time
    
    score_semantic = calculate_semantic_score(gt, bot_answer)
    score_rouge = calculate_rouge_l(gt, bot_answer)
    score_bleu = calculate_bleu(gt, bot_answer)
    score_f1 = calculate_token_f1(gt, bot_answer)
    
    retrieved_contexts = [f"[{i+1}] {doc.page_content[:150]}..." for i, doc in enumerate(child_docs)]
    
    report.append({
        "question": q,
        "ground_truth": gt,
        "bot_answer": bot_answer,
        "retrieved_contexts": retrieved_contexts,
        "latency": latency,
        "semantic": score_semantic,
        "rouge_l": score_rouge,
        "bleu": score_bleu,
        "f1": score_f1
    })
    print(f"Xong câu {idx + 1}/{len(test_dataset)} | Ngữ nghĩa: {score_semantic:.1f}%")

print("\n=== HOÀN TẤT CHẤM ĐIỂM! ĐANG DỰNG BÁO CÁO... ===")

# VẼ DASHBOARD BÁO CÁO HTML & CSS
avg_semantic = np.mean([res["semantic"] for res in report])
avg_rouge = np.mean([res["rouge_l"] for res in report])
avg_bleu = np.mean([res["bleu"] for res in report])
avg_f1 = np.mean([res["f1"] for res in report])
avg_latency = np.mean([res["latency"] for res in report])

html_content = f"""
<style>
    .report-container {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 20px 0; }}
    .report-title {{ color: #2c3e50; border-bottom: 3px solid #3498db; padding-bottom: 10px; margin-bottom: 20px; }}
    .academic-table {{ width: 100%; border-collapse: collapse; margin-bottom: 30px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); }}
    .academic-table th, .academic-table td {{ border: 1px solid #e2e8f0; padding: 12px 15px; text-align: center; font-size: 14px; }}
    .academic-table th {{ background-color: #2b6cb0; color: white; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 600; }}
    .academic-table tr:nth-child(even) {{ background-color: #f7fafc; }}
    .academic-table .tr-highlight {{ font-weight: bold; background-color: #ebf8ff !important; color: #2b6cb0; }}
    
    .card {{ background: #ffffff; border: 1px solid #e2e8f0; border-radius: 10px; padding: 20px; margin-bottom: 25px; box-shadow: 0 4px 6px rgba(0,0,0,0.02); }}
    .card-header {{ display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #edf2f7; padding-bottom: 10px; margin-bottom: 15px; }}
    .question-title {{ font-size: 15px; font-weight: bold; color: #2d3748; margin: 0; }}
    
    .metric-grid {{ display: flex; gap: 10px; margin: 12px 0; background: #f8fafc; padding: 10px; border-radius: 6px; border: 1px solid #edf2f7; }}
    .metric-item {{ flex: 1; text-align: center; font-size: 12px; font-weight: 600; color: #4a5568; }}
    .metric-val {{ font-size: 15px; font-weight: bold; color: #2b6cb0; margin-top: 3px; }}
    
    .compare-grid {{ display: flex; gap: 20px; margin-top: 15px; }}
    .compare-col {{ flex: 1; padding: 15px; border-radius: 8px; }}
    .col-truth {{ background-color: #f0fff4; border-left: 5px solid #38a169; }}
    .col-bot {{ background-color: #ebf8ff; border-left: 5px solid #3182ce; }}
    .col-title {{ font-weight: bold; font-size: 13px; margin-bottom: 8px; text-transform: uppercase; }}
    .col-content {{ font-size: 14px; line-height: 1.6; color: #2d3748; white-space: pre-line; }}
    
    .accordion {{ margin-top: 15px; border: 1px solid #e2e8f0; border-radius: 6px; }}
    .accordion summary {{ background: #f7fafc; padding: 8px 12px; font-size: 12px; font-weight: 600; color: #4a5568; cursor: pointer; }}
    .accordion-content {{ padding: 12px; background: #fff; font-size: 12px; color: #718096; }}
</style>

<div class="report-container">
    <h2 class="report-title">📝 BÁO CÁO ĐÁNH GIÁ CHẤT LƯỢNG RAG (ACADEMIC REPORT)</h2>
    
    <table class="academic-table">
        <thead>
            <tr>
                <th>Hệ Thống (System)</th>
                <th>Semantic Similarity (SBERT)</th>
                <th>ROUGE-L (LCS-F1)</th>
                <th>BLEU Score (BLEU-2)</th>
                <th>Token F1-Score (SQuAD)</th>
                <th>Avg Latency (Tốc độ)</th>
            </tr>
        </thead>
        <tbody>
            <tr class="tr-highlight">
                <td><b>Your Custom Parent-Child RAG (Qwen2.5)</b></td>
                <td>{avg_semantic:.2f}%</td>
                <td>{avg_rouge:.2f}%</td>
                <td>{avg_bleu:.2f}%</td>
                <td>{avg_f1:.2f}%</td>
                <td>{avg_latency:.2f}s</td>
            </tr>
        </tbody>
    </table>
"""

for idx, res in enumerate(report):
    context_html = "".join([f'<div style="margin-bottom: 6px;"><b>Đoạn [{i+1}]:</b> {ctx}</div>' for i, ctx in enumerate(res["retrieved_contexts"])])
    
    html_content += f"""
    <div class="card">
        <div class="card-header">
            <h4 class="question-title">CÂU HỎI {idx + 1}: {res['question']}</h4>
            <span style="font-size: 12px; color: #718096; font-weight: bold;">⏱️ Latency: {res['latency']:.2f}s</span>
        </div>
        
        <div class="metric-grid">
            <div class="metric-item">
                Semantic Match
                <div class="metric-val" style="color: #2b6cb0;">{res['semantic']:.1f}%</div>
            </div>
            <div class="metric-item" style="border-left: 1px solid #edf2f7; border-right: 1px solid #edf2f7;">
                ROUGE-L
                <div class="metric-val" style="color: #38a169;">{res['rouge_l']:.1f}%</div>
            </div>
            <div class="metric-item" style="border-right: 1px solid #edf2f7;">
                BLEU-2
                <div class="metric-val" style="color: #d69e2e;">{res['bleu']:.1f}%</div>
            </div>
            <div class="metric-item">
                Token F1
                <div class="metric-val" style="color: #e53e3e;">{res['f1']:.1f}%</div>
            </div>
        </div>
        
        <details class="accordion">
            <summary>🔍 Xem tài liệu mà VectorDB đã tìm được để đưa cho AI đọc</summary>
            <div class="accordion-content">
                {context_html}
            </div>
        </details>
        
        <div class="compare-grid">
            <div class="compare-col col-truth">
                <div class="col-title" style="color: #276749;">🎯 Tài liệu chuẩn (Ground Truth)</div>
                <div class="col-content">{res['ground_truth']}</div>
            </div>
            <div class="compare-col col-bot">
                <div class="col-title" style="color: #2b6cb0;">🤖 AI thực tế (Bot Answer)</div>
                <div class="col-content">{res['bot_answer']}</div>
            </div>
        </div>
    </div>
    """

html_content += "</div>"
display(HTML(html_content))

=== ĐANG CHẠY CHẤM ĐIỂM TRÊN 100 CÂU HỎI ===


Both `max_new_tokens` (=450) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Xong câu 1/100 | Ngữ nghĩa: 93.3%


Both `max_new_tokens` (=450) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=450) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Xong câu 2/100 | Ngữ nghĩa: 96.7%
Xong câu 3/100 | Ngữ nghĩa: 92.5%


...

Both `max_new_tokens` (=450) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=450) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Xong câu 98/100 | Ngữ nghĩa: 89.8%
Xong câu 99/100 | Ngữ nghĩa: 96.5%


Both `max_new_tokens` (=450) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Xong câu 100/100 | Ngữ nghĩa: 97.6%

=== HOÀN TẤT CHẤM ĐIỂM! ĐANG DỰNG BÁO CÁO... ===


Hệ Thống (System),Semantic Similarity (SBERT),ROUGE-L (LCS-F1),BLEU Score (BLEU-2),Token F1-Score (SQuAD),Avg Latency (Tốc độ)
Your Custom Parent-Child RAG (Qwen2.5),85.00%,67.93%,62.42%,55.01%,9.85s
